### 02 model: rgcn encoder + distmult decoder
reads: nothing  
writes: nothing (classes are copied into 03, 04)

#### imports

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

#### relational graph conv layer (one weight matrix per edge type, mean over neighbours)

In [2]:
class RelConv(nn.Module):
    def __init__(self, in_dim, out_dim, n_rel):
        super().__init__()
        self.W_rel  = nn.Parameter(torch.randn(n_rel, in_dim, out_dim) * 0.01)
        self.W_self = nn.Parameter(torch.randn(in_dim, out_dim) * 0.01)
        self.bias   = nn.Parameter(torch.zeros(out_dim))

    def forward(self, x, edge_sets):
        # edge_sets: list of (src, dst) tensors, one per relation, same order as W_rel
        out = x @ self.W_self
        for r, (src, dst) in enumerate(edge_sets):
            if src.numel() == 0:
                continue
            m = x[src] @ self.W_rel[r]
            agg = torch.zeros(x.size(0), out.size(1), device=x.device, dtype=out.dtype)
            agg.index_add_(0, dst, m)
            deg = torch.zeros(x.size(0), device=x.device)
            deg.index_add_(0, dst, torch.ones_like(dst, dtype=torch.float))
            deg = deg.clamp(min=1).unsqueeze(1)
            out = out + agg / deg
        return out + self.bias

#### encoder: project each node type to 128 dims, then 2 conv layers

In [3]:
class Encoder(nn.Module):
    def __init__(self, n_h, n_c, n_t, d_in_c, d_in_t, d=128, n_rel=4, dropout=0.2):
        super().__init__()
        self.herb_emb = nn.Embedding(n_h, d)
        self.proj_c = nn.Linear(d_in_c, d)
        self.proj_t = nn.Linear(d_in_t, d)
        self.l1 = RelConv(d, d, n_rel)
        self.l2 = RelConv(d, d, n_rel)
        self.drop = nn.Dropout(dropout)
        self.n_h, self.n_c, self.n_t = n_h, n_c, n_t

    def project(self, C_feat, T_feat):
        h = self.herb_emb.weight
        c = F.relu(self.proj_c(C_feat))
        t = F.relu(self.proj_t(T_feat))
        return torch.cat([h, c, t], dim=0)

    def forward(self, C_feat, T_feat, edge_sets):
        x = self.project(C_feat, T_feat)
        x = F.relu(self.l1(x, edge_sets))
        x = self.drop(x)
        x = self.l2(x, edge_sets)
        return x

#### decoder: distmult score for a compound-target pair

In [4]:
class DistMult(nn.Module):
    def __init__(self, d, n_rel=1):
        super().__init__()
        self.rel = nn.Parameter(torch.randn(n_rel, d) * 0.01)

    def forward(self, h_src, h_dst, r=0):
        return (h_src * self.rel[r] * h_dst).sum(dim=-1)

#### full model

In [5]:
class Model(nn.Module):
    def __init__(self, n_h, n_c, n_t, d_in_c, d_in_t, d=128):
        super().__init__()
        self.enc = Encoder(n_h, n_c, n_t, d_in_c, d_in_t, d=d, n_rel=4)
        self.dec = DistMult(d, n_rel=1)
        self.d = d

    def encode(self, C_feat, T_feat, edge_sets):
        return self.enc(C_feat, T_feat, edge_sets)

    def score_ct(self, Z, c_idx, t_idx):
        # c_idx is absolute compound index (0..n_c-1), t_idx is absolute target index
        n_h, n_c = self.enc.n_h, self.enc.n_c
        h_c = Z[n_h + c_idx]
        h_t = Z[n_h + n_c + t_idx]
        return self.dec(h_c, h_t)

#### sanity check on random data

In [6]:
# sanity
if __name__ == '__main__':
    n_h, n_c, n_t = 62, 2319, 1341
    Cf = torch.randn(n_c, 2048)
    Tf = torch.randn(n_t, 20)
    m = Model(n_h, n_c, n_t, 2048, 20)
    src = torch.randint(0, n_h, (100,))
    dst = torch.randint(n_h, n_h+n_c, (100,))
    esets = [(src, dst), (dst, src), (torch.tensor([]).long(), torch.tensor([]).long()), (torch.tensor([]).long(), torch.tensor([]).long())]
    Z = m.encode(Cf, Tf, esets)
    s = m.score_ct(Z, torch.tensor([0,1,2]), torch.tensor([5,6,7]))
    print(Z.shape, s.shape, s)

torch.Size([3722, 128]) torch.Size([3]) tensor([ 9.8462e-07, -1.8494e-06, -4.2495e-06], grad_fn=<SumBackward1>)
